## **Aim**
To implement a program that detects unauthorized file modifications by comparing current hash values with stored baseline hash values.

## **Algorithm**
**Step 1:** Import `hashlib`, `json`, `os`, and `datetime` libraries.

**Step 2:** Define a function `calculate_hash(filepath)` to compute SHA-256 hash of a file.

**Step 3:** Define `create_baseline(directory, baseline_file)` to walk through a directory, compute hashes for all files, and save them to a JSON baseline file with timestamps.

**Step 4:** Define `verify_integrity(baseline_file)` to load the baseline, recompute current hashes, and compare against stored values.

**Step 5:** Report files that are: unchanged, modified (hash mismatch), new (not in baseline), or deleted (in baseline but missing).

**Step 6:** In `main()`, create test files, generate baseline, modify a file, add a new file, delete a file, then run verification.

In [1]:
import hashlib
import json
import os
import shutil
from datetime import datetime

def calculate_hash(filepath):
    sha256 = hashlib.sha256()
    try:
        with open(filepath, "rb") as f:
            for chunk in iter(lambda: f.read(4096), b""):
                sha256.update(chunk)
        return sha256.hexdigest()
    except Exception:
        return None

def create_baseline(directory, baseline_file):
    baseline = {"created": datetime.now().isoformat(), "files": {}}
    for root, _, files in os.walk(directory):
        for file in files:
            filepath = os.path.join(root, file)
            rel_path = os.path.relpath(filepath, directory)
            file_hash = calculate_hash(filepath)
            if file_hash:
                stat = os.stat(filepath)
                baseline["files"][rel_path] = {
                    "hash": file_hash,
                    "size": stat.st_size,
                    "mtime": stat.st_mtime
                }
    with open(baseline_file, "w") as f:
        json.dump(baseline, f, indent=2)
    print(f"Baseline created: {baseline_file} with {len(baseline['files'])} files")

def verify_integrity(baseline_file, directory):
    with open(baseline_file, "r") as f:
        baseline = json.load(f)
    
    print(f"\n--- Integrity Verification ---")
    print(f"Baseline created: {baseline['created']}")
    print(f"Checking directory: {directory}\n")
    
    current_files = {}
    for root, _, files in os.walk(directory):
        for file in files:
            filepath = os.path.join(root, file)
            rel_path = os.path.relpath(filepath, directory)
            current_files[rel_path] = calculate_hash(filepath)
    
    baseline_files = set(baseline["files"].keys())
    current_file_set = set(current_files.keys())
    
    # Unchanged
    for f in sorted(baseline_files & current_file_set):
        if baseline["files"][f]["hash"] == current_files[f]:
            print(f"  [OK]     {f}")
    
    # Modified
    for f in sorted(baseline_files & current_file_set):
        if baseline["files"][f]["hash"] != current_files[f]:
            print(f"  [MODIFIED] {f} - Hash mismatch!")
    
    # New files
    for f in sorted(current_file_set - baseline_files):
        print(f"  [NEW]     {f} - Not in baseline")
    
    # Deleted files
    for f in sorted(baseline_files - current_file_set):
        print(f"  [DELETED] {f} - Missing from filesystem")

def main():
    test_dir = "./integrity_test"
    baseline_file = "baseline.json"
    
    if os.path.exists(test_dir):
        shutil.rmtree(test_dir)
    os.makedirs(test_dir)
    
    # Create test files
    with open(os.path.join(test_dir, "doc1.txt"), "w") as f:
        f.write("Important document v1")
    with open(os.path.join(test_dir, "doc2.txt"), "w") as f:
        f.write("Configuration data")
    with open(os.path.join(test_dir, "log.txt"), "w") as f:
        f.write("System log entry")
    
    create_baseline(test_dir, baseline_file)
    
    # Simulate changes
    print("\n--- Simulating unauthorized changes ---")
    with open(os.path.join(test_dir, "doc1.txt"), "w") as f:
        f.write("Important document v1 - TAMPERED")
    with open(os.path.join(test_dir, "new_file.txt"), "w") as f:
        f.write("Malicious payload")
    os.remove(os.path.join(test_dir, "log.txt"))
    
    verify_integrity(baseline_file, test_dir)

if __name__ == "__main__":
    main()

Baseline created: baseline.json with 3 files

--- Simulating unauthorized changes ---

--- Integrity Verification ---
Baseline created: 2026-08-20T09:00:06.648285
Checking directory: ./integrity_test

  [OK]     doc2.txt
  [MODIFIED] doc1.txt - Hash mismatch!
  [NEW]     new_file.txt - Not in baseline
  [DELETED] log.txt - Missing from filesystem


## **Result**
This the program successfully detects unauthorized file modifications by comparing current hash values with stored baseline hash values.